In [1]:
%pip install websockets

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install websockets cryptography

import asyncio
import websockets
import json
import time
from datetime import datetime

# NAZ'IN GÜVENLİK MODÜLÜNÜ İÇERİ AKTARIYORUZ
from security_layer import open_secure_packet, verify_signature, hash_password

Note: you may need to restart the kernel to use updated packages.


1) STATE MANAGEMENT ( SUNUCU HAFIZASI )

Bu kısımda aktif öğrenciler için tutacağımız sözlüğü açıyoruz. Anlık durumlar, webscoket kanalı ve şifresi burada tutulacak.

Örnek yapı bu şekilde: {"2300005352": {"ws": <baglanti_kanali>, "state": "in_progress", "session_token": "abc123xyz"}}

In [3]:
active_students = {}

Burada da sınav kayıt defteri var. Sınavların tipleri,süreleri ve bazı özellikleri çeşitli olacağı için onları burada tutacağız ( süreleri,kuralları vs.)

Örnek yapı: {"CS301": {"duration": 60, "prohibited": ["chatgpt"]}}

In [4]:
exam_registry = {}

In [5]:
dashboard_counter = 0  

In [6]:
print("Kütüphaneler yüklendi ve sunucu hafızası oluşturuldu.")

Kütüphaneler yüklendi ve sunucu hafızası oluşturuldu.


2) İLETİŞİM VE KOMUT YÖNETİMİ

Bu kısımda, handle_client fonksiyonu ile, JSON paketleri sunucuya ulaştığında "action" kelimesine göre o anki yapılan işlem ayırt edilerek ilgili işlemler yapılacak.

In [7]:
async def handle_client(websocket):
    global dashboard_counter
    try:
        async for message in websocket:
            data = json.loads(message)

            # 🛡️ YENİ: NAZ'IN ŞİFRELİ PAKETİ GELDİYSE ÖNCE AÇ!
            if "encrypted" in data and "signature" in data:
                try:
                    # open_secure_packet hem imzayı doğrular, hem zamanı (replay attack) kontrol eder, hem şifreyi çözer
                    data = open_secure_packet(message) 
                except Exception as e:
                    print(f"🚫 [SİBER GÜVENLİK] Geçersiz veya hacklenmiş paket reddedildi! Sebep: {e}")
                    continue # Kötü niyetli paketi işleme alma, döngüyü atla

            action = data.get("action")
            student_id = data.get("student_id") # ID'yi erkenden alıyoruz

            # RATE LIMITING: Aynı öğrenciden çok hızlı paket gelirse reddet
            if student_id and student_id in active_students:
                now = time.time()
                last_time = active_students[student_id].get("last_msg_time", 0)
                if now - last_time < 0.5:
                    continue 
                active_students[student_id]["last_msg_time"] = now

            if action == "register_exam": # EĞİTMENİN YENİ SINAV KAYIT KOMUTU
                exam_id = data["payload"]["exam_id"]
                exam_registry[exam_id] = data["payload"]
                print(f"\n✅ [EĞİTMEN] Yeni Sınav Aktif Edildi: {exam_id}") 
                await websocket.send(json.dumps({"status": "exam_registered"}))

            elif action == "resume_student": # EĞİTMENİN KOPAN ÖĞRENCİYİ AFFETME VE DEVAM ETTİRME KOMUTU
                # 🛡️ GEÇİCİ GÜVENLİK (Placeholder): Eğitmen rolü kontrolü
                if data.get("role") != "teacher":
                    print(f"🚫 [GÜVENLİK İHLALİ] Öğrenci kendini affettirmeye çalıştı! İSTEK REDDEDİLDİ.")
                    continue
                    
                hedef_id = data.get("student_id")
                if hedef_id in active_students:
                    active_students[hedef_id]["state"] = "in_progress"
                    print(f"\n🟢 [EĞİTMEN KOMUTU] {hedef_id} numaralı öğrenci affedildi ve DEVAM ETTİRİLDİ.")

            elif action == "request_start_exam": # ÖĞRENCİNİN SINAVA BAŞLAMA VEYA DEVAM ETME TALEBİ


                # 🛡️ 1. MESAJ BÜTÜNLÜĞÜ (IMZA) KONTROLÜ
                client_sig = data.pop("auth_signature", None)
                if client_sig:
                    msg_str = json.dumps(data, sort_keys=True)
                    if not verify_signature(msg_str, client_sig):
                        await websocket.send(json.dumps({"status": "error", "message": "Güvenlik İhlali: Mesaj İmzası Geçersiz!"}))
                        print(f"🚫 [SİBER GÜVENLİK] {data.get('student_id')} için sahte imza tespit edildi!")
                        continue
                else:
                    print("⚠️ [UYARI] Eski sürüm bir istemci bağlanmaya çalışıyor (İmza yok).")

                # 🛡️ 2. KİMLİK (CREDENTIAL) DOĞRULAMA
                login_id = data.get("login_id", "")
                gelen_sifre = data.get("password", "")
                gelen_hash = data.get("password_hash", "")
                credential_sig = data.get("credential_sig", "")

                # 🛡️ YENİ EKLENEN: KİMLİK İMZASI (INTEGRITY) KONTROLÜ
                if login_id and gelen_hash and credential_sig:
                    beklenen_imza_metni = f"{login_id}:{gelen_hash}"
                    # Naz'ın modülündeki verify_signature fonksiyonunu kullanıyoruz
                    if not verify_signature(beklenen_imza_metni, credential_sig):
                        await websocket.send(json.dumps({"status": "error", "message": "Güvenlik İhlali: Kimlik İmzası Geçersiz!"}))
                        print(f"🚫 [SİBER GÜVENLİK] {data.get('student_id')} için sahte kimlik imzası (credential_sig) tespit edildi!")
                        continue
                if gelen_sifre and gelen_hash:
                    # Gelen düz şifreyi sunucuda hash'leyip, Naz'ın gönderdiği hash ile eşleşiyor mu bakıyoruz
                    beklenen_hash = hash_password(gelen_sifre)
                    if beklenen_hash != gelen_hash:
                        await websocket.send(json.dumps({"status": "error", "message": "Hatalı Şifre!"}))
                        print(f"🚫 [SİBER GÜVENLİK] {data.get('student_id')} için Hatalı Şifre girişimi!")
                        continue


                student_id = data["student_id"]
                exam_id = data["exam_id"]
                
               

                # AŞAĞIYI ŞİMDİLİK YORUM SATIRI YAPTIM, ÇÜNKÜ SİSTEMİ NASIL YAPACAĞIMIZA KARAR VEREMEDİK. İLERDE ÖĞRENCİNİN KAYITLI SINAVLAR ARASINDAN SEÇMESİ GİBİ BİR ŞEY EKLEMEK İSTİYORUZ.

               # if exam_id not in exam_registry: # Sınav ID'si geçerli mi kontrolü
               #     await websocket.send(json.dumps({"status": "error", "message": f"HATA: '{exam_id}' bulunamadı!"}))
               #     continue
                
                # ÇOKLU GİRİŞ VE FARKLI SINAV KONTROLÜ 
                if student_id in active_students:
                    mevcut_durum = active_students[student_id]["state"]
                    mevcut_sinav = active_students[student_id]["exam_id"]

                    # Kural 1: Öğrenci zaten içeride ve sınavı devam ediyorsa yeni cihazı REDDET!
                    if mevcut_durum == "in_progress":
                        await websocket.send(json.dumps({
                            "status": "error", 
                            "message": f"HATA: Zaten '{mevcut_sinav}' sınavındasınız. Aynı anda iki yerden girilemez!"
                        }))
                        continue
                    
                    # Kural 2: Kopmuş/Dondurulmuş ama FARKLI sınava girmeye çalışıyor -> REDDET!
                    elif mevcut_durum in ["disconnected_paused", "violation_paused"] and mevcut_sinav != exam_id:
                        await websocket.send(json.dumps({
                            "status": "error", 
                            "message": f"HATA: Önce '{mevcut_sinav}' sınavını bitirmelisiniz. '{exam_id}' sınavına geçemezsiniz!"
                        }))
                        continue

                    # ESKİ HALİ: Kopmuş/Dondurulmuş ve AYNI sınava girmeye çalışıyor -> KURTAR
                    
                    # YENİ KURAL 3: Sadece BAĞLANTISI KOPAN (disconnected) masum öğrenciyi kurtar (Reconnect)
                    elif mevcut_durum == "disconnected_paused" and mevcut_sinav == exam_id:
                        kalan_saniye = active_students[student_id]["time_left"]
                        session_token = active_students[student_id]["session_token"]
                        
                        active_students[student_id]["ws"] = websocket
                        active_students[student_id]["state"] = "in_progress"
                        
                        print(f"\n🔄 [BİLGİ] {student_id} bağlantısı koptuğu yerden tekrar içeri alındı!")
                        await websocket.send(json.dumps({
                            "action": "exam_started_ack",
                            "status": "success",
                            "session_token": session_token,
                            "reconnected": True,
                            "time_left_seconds": kalan_saniye
                        }))
                        continue
                        
                    # YENİ KURAL 4: KOPYA ÇEKEN (violation) öğrenci reconnect yapmaya çalışırsa -> REDDET!
                    elif mevcut_durum == "violation_paused" and mevcut_sinav == exam_id:
                        # Bağlantı objesini yenile (Hoca affederse komut gidebilsin diye) ama durumunu değiştirme!
                        active_students[student_id]["ws"] = websocket 
                        
                        print(f"\n🚫 [GÜVENLİK] {student_id} ihlalden dondurulduğu için reconnect isteği REDDEDİLDİ.")
                        await websocket.send(json.dumps({
                            "status": "error", 
                            "message": "Sınavınız güvenlik ihlali sebebiyle hocanız tarafından durdurulmuştur. Lütfen gözetmen ile iletişime geçiniz."
                        }))
                        continue
                # ----------------------------------------------------------------------
                
                # İlk kez giriyorsa normal başlat
                # Yeni Hali (Mert'in DB'si gelene kadar varsayılan 40 dakika veriyoruz):
                duration_mins = 40
                duration_secs = duration_mins * 60
                session_token = f"token_{student_id}_gizli"

                # 🛡️ 3. KİMLİK BİLGİLERİNİ RAM'E KAYDET (YENİ EKLENDİ)
                active_students[student_id] = {  
                    "ws": websocket,
                    "state": "in_progress",
                    "session_token": session_token,
                    "exam_id": exam_id,
                    "time_left": duration_secs,
                    # Naz'ın gönderdiği yeni verileri ileride kullanmak üzere saklıyoruz:
                    "login_id": login_id,
                    "password_hash": gelen_hash,
                    "credential_sig": credential_sig
                }
                
                await websocket.send(json.dumps({ # Öğrenciye sınav başlama onayı ve oturum bilgisi gönder
                    "action": "exam_started_ack",
                    "status": "success",
                    "session_token": session_token,
                    "reconnected": False,
                    "total_duration_minutes": duration_mins 
                }))

            elif action == "status_update":  # ÖĞRENCİNİN SÜREÇ İÇİNDEKİ DURUM GÜNCELLEMESİ
                student_id = data.get("student_id")
                token = data.get("session_token")

                # Naz'ın auth modülüyle şifreli token doğrulaması buraya gelecek
                if student_id in active_students and active_students[student_id]["session_token"] == token:
                    
                    security_data = data.get("security", {})
                    
                    # Ahmet'ten gelen ihlal uyarısı True ise detayları işle
                    if security_data.get("violation_alert") == True:
                        active_students[student_id]["state"] = "violation_paused"
                        
                        # Ahmet ve Engin'in gönderdiği detayları ayıklıyoruz
                        v_type = security_data.get("violation_type", "Bilinmeyen İhlal")
                        details = security_data.get("details", {})
                        aktif_pencere = details.get("active_window", "Bilinmiyor")
                        acik_uygulamalar = details.get("open_apps", [])
                        
                        # 🧠 DİNAMİK VE BİRİKİMLİ ŞÜPHE SKORLAMASI
                        mevcut_skor = active_students[student_id].get("total_risk_score", 0)
                        ek_skor = 0
                        high_risk_words = ["chatgpt", "discord", "whatsapp", "telegram", "gemini", "claude", "chegg", "stackoverflow"]
                        medium_risk_words = ["google", "bing", "brave", "search", "yandex"]
                        
                        tum_uygulamalar_str = (aktif_pencere + " " + " ".join(acik_uygulamalar)).lower()
                        
                        for word in high_risk_words:
                            if word in tum_uygulamalar_str:
                                ek_skor += 40
                        for word in medium_risk_words:
                            if word in tum_uygulamalar_str:
                                ek_skor += 15
                                
                        yeni_skor = min(mevcut_skor + ek_skor, 100)
                        
                        if yeni_skor >= 80:
                            risk_level = "KRİTİK"
                        elif yeni_skor >= 40:
                            risk_level = "YÜKSEK"
                        else:
                            risk_level = "ORTA"
                            
                        # Dashboard'un okuyabilmesi için doğrudan öğrenci hafızasına yazıyoruz
                        active_students[student_id]["total_risk_score"] = yeni_skor
                        active_students[student_id]["risk_level"] = risk_level
                        
                        # Engin'in saatini boşverip sunucunun ISO zamanını basıyoruz
                        zaman = datetime.now().isoformat()
                        
                        # İrem ve Rana'nın Dashboard'da göstermesi için RAM'e kaydediyoruz
                        active_students[student_id]["last_violation"] = {
                            "type": v_type,
                            "window": aktif_pencere,
                            "time": zaman,
                            "risk_score": yeni_skor,
                            "risk_level": risk_level
                        }
                        
                        print(f"\n🚨 [ALARM] {student_id} ihlal yaptı! Sınav donduruldu.")
                        print(f"   ↳ 🧠 [ANALİZ] Güvenlik Skoru: %{yeni_skor} - Seviye: {risk_level}")
                        print(f"   ↳ Sebep: {v_type} | Pencere: {aktif_pencere}")
                        print(f"   ↳ Arka Plan: {', '.join(acik_uygulamalar)}")
                        
                        # Mert'in DB modülü bittiğinde kayıt fonksiyonu buraya eklenecek
                        
                        # Mert'in DB modülü bittiğinde kayıt fonksiyonu buraya eklenecek
                        # db.save_violation_to_sql(student_id, v_type, aktif_pencere)
 
            elif action == "get_dashboard_data":
                dashboard_counter += 1
                
                # İrem ve Rana için veriyi sunucu tarafında hazırlıyoruz
                formatted_students = {}
                for sid, info in active_students.items():
                    raw_seconds = info.get("time_left", 0)
                    
                    # Saniyeyi "DAKİKA:SANİYE" formatına çeviriyoruz
                    mins = raw_seconds // 60
                    secs = raw_seconds % 60
                    time_str = f"{mins:02d}:{secs:02d}" # Örn: "14:05"

                    # Sınavı bitenleri eğitmen panelinden gizle
                    if info["state"] == "completed":
                        continue
                    
                    # Dashboard'a giden veri paketine risk bilgilerini ekliyoruz
                    formatted_students[sid] = {
                        "state": info["state"], 
                        "exam_id": info.get("exam_id"), 
                        "time_left_formatted": time_str, # Yeni
                        "risk_score": info.get("total_risk_score", 0), # Yeni
                        "risk_level": info.get("risk_level", "TEMİZ")   # Yeni
                    }

                dashboard_data = {
                    "action": "dashboard_update",
                    "active_students_count": len(active_students),
                    "students": formatted_students
                }
                await websocket.send(json.dumps(dashboard_data))
                print(f"\r📊 [SİSTEM] Dashboard güncellendi. (İstek: {dashboard_counter})", end="", flush=True)

    except Exception as e:
        # Eski hali: pass  (Hataları gizliyordu)
        print(f"❌ [SİSTEM HATASI] Beklenmedik bir sorun oluştu: {e}")
    finally:
        for sid, info in active_students.items():
            if info["ws"] == websocket:
                info["ws"] = None # YENİ: Kopan öğrencinin bozuk ağ kablosunu RAM'den temizle
                
                # SADECE in_progress ise durumu değiştir
                if info["state"] == "in_progress":
                    if info.get("time_left", 1) <= 0:
                        info["state"] = "completed"
                        print(f"\n✅ [BİTTİ] Öğrenci {sid} süresini doldurdu ve sınavı tamamladı.")
                    else:
                        info["state"] = "disconnected_paused"
                        print(f"\n🔌 [KOPTU] Öğrenci {sid} hattan düştü. Durumu donduruldu.")
                break

print("✅ Sunucu güncellendi!")

✅ Sunucu güncellendi!


3) SUNUCUYU BAŞLATMA VE AÇIK TUTMA

In [8]:
print("🌐 [SUNUCU] Merkezi Sınav Sunucusu Başlatılıyor...")

async def global_timer():
    while True:
        await asyncio.sleep(1) # Her 1 saniyede bir tetiklenir

        for sid, info in active_students.items():
            if info["state"] == "in_progress":
                info["time_left"] -= 1 # Süreyi sunucuda azalt

                if info["time_left"] <= 0:
                    info["state"] = "completed"
                    print(f"\n✅ [BİTTİ] {sid} numaralı öğrencinin SÜRESİ DOLDU!")

async def run_server():
    
    global active_students, exam_registry, dashboard_counter
    active_students.clear()
    exam_registry.clear()
    dashboard_counter = 0
    
    print("🧹 [SİSTEM] Eski oturumlar, sınavlar ve loglar RAM'den tamamen silindi!")
    
    #  EKSİK OLAN SATIR: Sunucu sayacını arka planda başlat
    asyncio.create_task(global_timer())

    async with websockets.serve(handle_client, "0.0.0.0", 8765):
        print("🌐 [SUNUCU] Port 8765 dinleniyor. İstemciler bekleniyor...\n")
        try:
            await asyncio.Future()  
        except asyncio.CancelledError:
            print("🛑 [SİSTEM] Sunucu manuel olarak durduruldu. Kapatılıyor...")

await run_server()

🌐 [SUNUCU] Merkezi Sınav Sunucusu Başlatılıyor...
🧹 [SİSTEM] Eski oturumlar, sınavlar ve loglar RAM'den tamamen silindi!
🌐 [SUNUCU] Port 8765 dinleniyor. İstemciler bekleniyor...

🛑 [SİSTEM] Sunucu manuel olarak durduruldu. Kapatılıyor...
